In [ ]:
# ==========================================
# FAKE GOOGLE COLAB & OPENAI MOCK FOR LOCAL RUN
# ==========================================
import sys
from types import ModuleType
import os

# Set API keys
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
os.environ["OPENROUTER_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY", "")

class FakeUserdata:
    def get(self, key):
        if key == "GROQ_API_KEY":
            return os.environ.get("GROQ_API_KEY", "")
        if key == "HF_TOKEN":
            return os.environ.get("HF_TOKEN", "")
        if key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY"]:
            return os.environ.get("GROQ_API_KEY", "")
        return os.environ.get(key)

class FakeFiles:
    def upload(self):
        print("MOCK: files.upload() - using local 'anual report mini proj 01.pdf'")
        return {"anual report mini proj 01.pdf": b""}

# Inject google.colab
colab_module = ModuleType("google.colab")
colab_module.userdata = FakeUserdata()
colab_module.files = FakeFiles()
sys.modules["google.colab"] = colab_module

# Patch openai to redirect to Groq (with recursion guard)
import openai
from openai import OpenAI

if not getattr(OpenAI, "_is_patched", False):
    orig_init = OpenAI.__init__
    def new_init(self, *args, **kwargs):
        kwargs["api_key"] = os.environ.get("GROQ_API_KEY", "")
        kwargs["base_url"] = "https://api.groq.com/openai/v1"
        orig_init(self, *args, **kwargs)
    OpenAI.__init__ = new_init
    OpenAI._is_patched = True

# Map models in Chat Completions (with recursion guard)
from openai.resources.chat.completions import Completions

if not getattr(Completions, "_is_patched", False):
    orig_create = Completions.create
    def new_create(self, *args, **kwargs):
        model = kwargs.get("model")
        if model in ["gpt-4o", "gpt-4o-mini", "gpt-4-turbo", "openai/gpt-4o", "openai/gpt-4o-mini"]:
            kwargs["model"] = "llama-3.3-70b-versatile"
        elif model in ["llama-3.1-8b-instant", "meta-llama/llama-3.1-8b-instant"]:
            kwargs["model"] = "llama-3.1-8b-instant"
        
        # If the response format is JSON, force compatible settings
        if "response_format" in kwargs and isinstance(kwargs["response_format"], dict):
            if kwargs["response_format"].get("type") == "json_object":
                pass # Keep it
                
        return orig_create(self, *args, **kwargs)
    Completions.create = new_create
    Completions._is_patched = True

print("MOCK: google.colab and openai patched successfully!")


# Install Libraries

In [2]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-text-splitters
!pip install -q pymupdf
!pip install -q openai
!pip install -q pandas
!pip install -q tqdm
!pip install -q groq

zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


zsh:1: command not found: pip


# Import Libraries

In [3]:
import fitz
import json
import random
import pandas as pd

from tqdm import tqdm
from groq import Groq

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Read PDF

In [5]:
!pip install -q pymupdf # Ensure pymupdf is installed
import fitz # Added import statement

pdf_path = "anual report mini proj 01.pdf" # Corrected filename

doc = fitz.open(pdf_path)

print("Pages:", len(doc))

zsh:1: command not found: pip


Pages: 142


# Extract Text

In [6]:
all_text = ""

for page in doc:

    text = page.get_text()

    all_text += text + "\n"

print(all_text[:1000])

On Our Way
2024 ANNUAL REPORT

Uber’s Mission
We reimagine the way the world moves for the better
We are Uber. The go-getters. The kind of people who are relentless about our 
mission to help people go anywhere and get anything and earn their way. 
Movement is what we power. It’s our lifeblood. It runs through our veins. It’s 
what gets us out of bed each morning. It pushes us to constantly reimagine 
how we can move better. For you. For all the places you want to go. For all the 
things you want to get. For all the ways you want to earn. Across the entire 
world. In real time. At the incredible speed of now.

 
 
UNITED STATES 
SECURITIES AND EXCHANGE COMMISSION 
Washington, D.C. 20549 
____________________________________________  
FORM 10-K 
____________________________________________  
(Mark One) 
 
☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 
For the fiscal year ended December 31, 2024 
OR 
☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 1

# Clean Text

In [7]:
def clean_text(text):

    text = text.replace("\n", " ")

    text = text.replace("  ", " ")

    text = text.strip()

    return text

cleaned_text = clean_text(all_text)

print(cleaned_text[:1000])

On Our Way 2024 ANNUAL REPORT Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.   UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________  FORM 10-K ____________________________________________  (Mark One)  ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2024 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIE

In [8]:
print(cleaned_text[:500])

On Our Way 2024 ANNUAL REPORT Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to g


# Chunking

In [9]:
!pip install -q langchain-text-splitters # Ensure package is installed
from langchain_text_splitters import RecursiveCharacterTextSplitter # Added import
splitter = RecursiveCharacterTextSplitter(

    chunk_size=1500,

    chunk_overlap=200

)

chunks = splitter.split_text(cleaned_text)

print(len(chunks))

zsh:1: command not found: pip


481


In [10]:
print(chunks[0])

On Our Way 2024 ANNUAL REPORT Uber’s Mission We reimagine the way the world moves for the better We are Uber. The go-getters. The kind of people who are relentless about our mission to help people go anywhere and get anything and earn their way. Movement is what we power. It’s our lifeblood. It runs through our veins. It’s what gets us out of bed each morning. It pushes us to constantly reimagine how we can move better. For you. For all the places you want to go. For all the things you want to get. For all the ways you want to earn. Across the entire world. In real time. At the incredible speed of now.   UNITED STATES SECURITIES AND EXCHANGE COMMISSION Washington, D.C. 20549 ____________________________________________  FORM 10-K ____________________________________________  (Mark One)  ☒ ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934 For the fiscal year ended December 31, 2024 OR ☐ TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIE

In [11]:
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

In [12]:
from google.colab import userdata
userdata.get('GROQ_API_KEY')

'YOUR_GROQ_API_KEY'

In [13]:
from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Define the 'prompt' variable.
prompt = "What is Uber's mission?"

# Using 'llama-3.1-8b-instant' as previous llama3 models are decommissioned
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

questions = response.choices[0].message.content
print(questions)

Uber's mission is to "Transport your World". However, their initial public mission statement was "To long-term benefit of society, why should we own cars in 10 years?"


In [14]:
response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)

questions = response.choices[0].message.content
print(questions)

Uber's mission is to create mobility for all, which encompasses connecting people and things through reliable, energy-efficient, and safe transportation solutions, food delivery, logistics, and other services. Their main goals are:

1. To make transportation affordable and accessible to everyone, regardless of income, geography, or social status.
2. To create a sustainable transportation system that minimizes the environmental impact of cars, trucks, and other vehicles.
3. To provide a safe and reliable transportation experience for passengers, with features such as driver verification, background checks, and vehicle inspections.

By achieving these goals, Uber aims to transform the way people move around cities and make transportation a seamless, efficient, and environmentally friendly experience.


In [15]:
from google.colab import userdata
from groq import Groq

# Ensure client is initialized
client = Groq(api_key=userdata.get('GROQ_API_KEY'))

# Re-fetch the mission statement to ensure 'questions' is defined
prompt = "What is Uber's mission?"
response_mission = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)
questions = response_mission.choices[0].message.content

# Now generate the summary
answer_prompt = f"Please provide a one-sentence summary of this mission statement: {questions}"

response_summary = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": answer_prompt}]
)

answers = response_summary.choices[0].message.content
print(f"Original Mission: {questions}\n")
print(f"Summary: {answers}")

Original Mission: Uber's mission is to "transport your world" but according to Uber's website their stated mission is to "create opportunities through movement" this is by connecting drivers with passengers and also facilitating the movement of goods through their logistics service Uber Freight and food delivery with Uber Eats

Summary: Uber's mission is to create opportunities through movement by connecting people, goods, and services via transportation, logistics, and food delivery services.


# MASTER PROMPT  — Question Generation


In [16]:
# MASTER PROMPT A — Question Generation


QUESTION_GENERATION_PROMPT = """You are a senior financial analyst at a quantitative hedge fund.
Your task is to generate exactly 10 high-quality questions based ONLY on the text provided below.

The questions must cover ALL THREE of these categories:
1. HARD FACTS (at least 4 questions): Ask about specific numbers, dollar amounts, percentages,
   dates, names of people, company names, or exact statistics mentioned in the text.
   Example: "What was Uber's total revenue for the fiscal year ended December 31, 2024?"

2. STRATEGIC SUMMARY (at least 3 questions): Ask about business strategy, competitive positioning,
   risks, market opportunities, or management's stated goals and plans.
   Example: "What are the primary competitive threats Uber identifies in its Mobility segment?"

3. STYLISTIC/CREATIVE (at least 3 questions): Ask for summaries written in a specific voice,
   investor memos, press releases, or rewritten explanations for a non-technical audience.
   Example: "Write a 3-sentence executive summary of this section for a non-technical board member."

RULES:
- Generate EXACTLY 10 questions — no more, no less
- Base questions STRICTLY on the provided text — do not invent information
- Label each question with its category: [HARD FACT], [STRATEGIC], or [CREATIVE]
- Output ONLY a numbered list (1. ... 2. ... etc.), nothing else

TEXT TO ANALYZE:
{chunk}

OUTPUT (10 numbered questions only):"""


# ──────────────────────────────────────────────────────────────────
# MASTER PROMPT B — Answer Generation
#
# ──────────────────────────────────────────────────────────────────

ANSWER_GENERATION_PROMPT = """You are a senior financial analyst at a quantitative hedge fund.
You have been given a passage from Uber's 2024 Annual Report and a set of questions about it.
Answer each question using ONLY information found in the provided passage.

RULES:
- Answer every question in order (1 through 10)
- For [HARD FACT] questions: be precise — include exact numbers, names, and dates from the text
- For [STRATEGIC] questions: write 2-4 sentences with clear reasoning
- For [CREATIVE] questions: write in the requested format and tone (memo, summary, press release)
- If the passage does not contain enough information to answer a question, write:
  "Insufficient information in this passage."
- Do NOT invent or hallucinate any facts not in the passage
- Output format: number each answer to match its question (1. ... 2. ... etc.)

PASSAGE:
{chunk}

QUESTIONS:
{questions}

ANSWERS (numbered 1-10):"""

print(" Master prompts defined!")
print(f"   Question prompt : {len(QUESTION_GENERATION_PROMPT)} characters")
print(f"   Answer prompt   : {len(ANSWER_GENERATION_PROMPT)} characters")


 Master prompts defined!
   Question prompt : 1386 characters
   Answer prompt   : 888 characters


# Generate Answers

In [17]:
def parse_numbered_list(text: str) -> list[str]:
    """
    Extract items from a numbered list like:
        1. First item
        2. Second item
    Returns a clean Python list of strings.
    """
    lines = text.strip().split('\n')
    items = []
    for line in lines:
        line = line.strip()
        # Match lines that start with a number + period or parenthesis
        match = re.match(r'^\d+[.)\s]+(.+)', line)
        if match:
            items.append(match.group(1).strip())
    return items


def generate_questions(chunk: str, max_retries: int = 3) -> list[str]:
    """
    Call LLM A to generate 10 questions from a chunk.

    Returns a list of question strings, or an empty list if it fails.
    max_retries: how many times to retry on API errors before giving up
    """
    prompt = QUESTION_GENERATION_PROMPT.format(chunk=chunk)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,     # some creativity in question variety
                max_tokens=1000,     # questions don't need to be long
            )
            raw_output = response.choices[0].message.content
            questions = parse_numbered_list(raw_output)

            # If we got fewer than 5 questions, something went wrong — retry
            if len(questions) < 5:
                print(f"Only got {len(questions)} questions, retrying... (attempt {attempt+1})")
                time.sleep(2)
                continue

            return questions[:10]  # cap at 10 just in case

        except Exception as e:
            print(f"API error on attempt {attempt+1}: {e}")
            time.sleep(3)

    return []  # return empty list if all retries fail


def generate_answers(chunk: str, questions: list[str], max_retries: int = 3) -> list[str]:
    """
    Call LLM B to answer all 10 questions using the chunk as context.

    Returns a list of answer strings.
    """
    # Format the questions as a numbered list for the prompt
    questions_text = "\n".join([f"{i+1}. {q}" for i, q in enumerate(questions)])
    prompt = ANSWER_GENERATION_PROMPT.format(chunk=chunk, questions=questions_text)

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.3,     # lower temperature = more factual, less creative
                max_tokens=2000,     # answers can be longer than questions
            )
            raw_output = response.choices[0].message.content
            answers = parse_numbered_list(raw_output)

            if len(answers) < 5:
                print(f" Only got {len(answers)} answers, retrying... (attempt {attempt+1})")
                time.sleep(2)
                continue

            return answers[:10]

        except Exception as e:
            print(f" API error on attempt {attempt+1}: {e}")
            time.sleep(3)

    return []


print(" Generation functions defined!")
print("   generate_questions() → calls LLM A → returns 10 questions")
print("   generate_answers()   → calls LLM B → returns 10 answers")


 Generation functions defined!
   generate_questions() → calls LLM A → returns 10 questions
   generate_answers()   → calls LLM B → returns 10 answers


# Genaration LOOP

In [18]:
from pathlib import Path # Import Path for directory handling
import json # Used for checkpointing
import time # Used for delays
import re # Used by parse_numbered_list in generate_questions/answers (from AEJ8CzDYj2C4)
from tqdm import tqdm # Used in the main loop
from groq import RateLimitError, APITimeoutError # Added to catch specific Groq errors

# Assume MODEL is defined elsewhere, or provide a default if not.
# Based on other cells, it's 'llama-3.1-8b-instant'
if 'MODEL' not in globals():
    MODEL = "llama-3.1-8b-instant"
    print("WARNING: MODEL not defined, using default 'llama-3.1-8b-instant'. Please define MODEL in an earlier cell if this is incorrect.")

# Assume _detect_category is defined elsewhere, or provide a dummy if not.
# This function is used to auto-label categories. A dummy implementation will prevent NameError.
if '_detect_category' not in globals():
    def _detect_category(question: str) -> str:
        if "[HARD FACT]" in question.upper():
            return "HARD FACT"
        elif "[STRATEGIC]" in question.upper():
            return "STRATEGIC"
        elif "[CREATIVE]" in question.upper():
            return "CREATIVE"
        else:
            return "UNKNOWN"
    print("WARNING: _detect_category not defined, using a placeholder. Please define _detect_category in an earlier cell if this is incorrect.")

# Assume generate_questions and generate_answers are defined elsewhere (e.g., in AEJ8CzDYj2C4).
# This cell uses them. If they are not defined, this cell will still fail when calling them.

# ──────────────────────────────────────────────────────────────────
# CONFIGURATION — tweak these before running
# ──────────────────────────────────────────────────────────────────

# Set to None to process ALL chunks
# Set to a small number like 10 for a quick test run first!
MAX_CHUNKS = 15       # e.g., MAX_CHUNKS = 10 for testing

# How many seconds to wait between chunks (avoids rate limits)
DELAY_BETWEEN_CHUNKS = 20.0

# Define the output directory
OUTPUT_DIR = Path('./output')
OUTPUT_DIR.mkdir(exist_ok=True) # Ensure the output directory exists

# File to save progress checkpoint (so we can resume if interrupted)
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.json"

# ──────────────────────────────────────────────────────────────────
# LOAD EXISTING CHECKPOINT (if resuming a previous run)
# ──────────────────────────────────────────────────────────────────
all_qa_pairs = []
start_idx = 0

if CHECKPOINT_FILE.exists():
    print(" Found checkpoint file — loading previous progress...")
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint_data = json.load(f)
    all_qa_pairs = checkpoint_data.get("pairs", [])
    start_idx = checkpoint_data.get("next_chunk_idx", 0)
    print(f"   Resuming from chunk {start_idx} with {len(all_qa_pairs)} pairs already saved")
else:
    print(" No checkpoint found — starting fresh")

# ──────────────────────────────────────────────────────────────────
# DETERMINE WHICH CHUNKS TO PROCESS
# ──────────────────────────────────────────────────────────────────
# Fix for NameError: 'chunks' is not defined
if 'chunks' not in globals():
    print(" ERROR: 'chunks' variable is not defined. Please ensure cell 5hA2tpxOn-Ty (chunking) has been executed to load the document chunks.")
    # Provide an empty list for 'chunks' to prevent further NameErrors.
    # The Q&A generation will not proceed meaningfully without actual data.
    chunks = []
    chunks_to_process = [] # No chunks to process if chunks is empty
else:
    chunks_to_process = chunks[start_idx:]
    if MAX_CHUNKS is not None:
        chunks_to_process = chunks_to_process[:MAX_CHUNKS]

print(f"\n Starting generation loop...")
print(f"   Total chunks available : {len(chunks)}")
print(f"   Chunks to process now  : {len(chunks_to_process)}")
print(f"   Starting from index    : {start_idx}")
print(f"   Model                  : {MODEL}")
print()

# ──────────────────────────────────────────────────────────────────
# MAIN LOOP
# ──────────────────────────────────────────────────────────────────
if not chunks_to_process: # Only iterate if there are chunks to process
    print("No chunks to process. Ensure 'chunks' variable is defined and not empty from previous cells (5hA2tpxOn-Ty).")
else:
    for i, chunk in enumerate(tqdm(chunks_to_process, desc="Generating Q&A pairs")):
        actual_idx = start_idx + i  # real index in the full chunks list

        try:
            # ── Step A: Generate 10 questions from this chunk ──
            if 'generate_questions' not in globals():
                print(f" ERROR: 'generate_questions' function is not defined. Please ensure cell AEJ8CzDYj2C4 has been executed.")
                questions = [] # Prevent further errors
            else:
                questions = generate_questions(chunk)

            if not questions:
                print(f"    Skipping chunk {actual_idx} — no questions generated")
                continue

            # ── Step B: Generate answers for those questions ──
            if 'generate_answers' not in globals():
                print(f"   ERROR: 'generate_answers' function is not defined. Please ensure cell AEJ8CzDYj2C4 has been executed.")
                answers = [] # Prevent further errors
            else:
                answers = generate_answers(chunk, questions)

            if not answers:
                print(f"   Skipping chunk {actual_idx} — no answers generated")
                continue

            # ── Pair up questions and answers, save each pair ──
            for j, (question, answer) in enumerate(zip(questions, answers)):
                if question.strip() and answer.strip():
                    all_qa_pairs.append({
                        "chunk_id"     : actual_idx,       # which chunk this came from
                        "pair_id"      : j,                # which Q&A pair within the chunk
                        "context"      : chunk,            # the source text (useful for RAG later)
                        "question"     : question,
                        "answer"       : answer,
                        "category"     : _detect_category(question),  # auto-label the category
                    })

            # ── Save checkpoint every 10 chunks ──
            if (i + 1) % 10 == 0:
                checkpoint = {"pairs": all_qa_pairs, "next_chunk_idx": actual_idx + 1}
                with open(CHECKPOINT_FILE, 'w') as f:
                    json.dump(checkpoint, f)

            # ── Wait between chunks to avoid rate limits ──
            time.sleep(DELAY_BETWEEN_CHUNKS)

        except RateLimitError as e:
            print(f"\n Rate Limit Exceeded for chunk {actual_idx}: {e}")
            message = str(e)
            match = re.search(r"Please try again in (\d+\.?\d*)s", message)
            if match:
                wait_time = float(match.group(1))
                wait_time_with_buffer = wait_time + 10 # Add 10 second buffer
                print(f"  Waiting for {wait_time_with_buffer:.2f} seconds before retrying this chunk...")
                time.sleep(wait_time_with_buffer)
                print("  This might be a daily limit. It's recommended to resume tomorrow if the error persists.")
                # After waiting, the loop will try the *same* chunk again since it's inside the for loop iteration.
                # If it's truly a daily limit, breaking is more appropriate than retrying the same chunk repeatedly.
                break # Break out of the loop as a daily limit is likely hit.
            else:
                print("  Could not parse specific wait time from error message. Waiting for 5 minutes and then stopping.")
                time.sleep(300) # Wait 5 minutes
                print("  The rate limit is often daily. It's recommended to resume tomorrow.")
                break # Stop processing for the current run as daily limit is hit

        except APITimeoutError as e:
            print(f" API Timeout Error for chunk {actual_idx}: {e}")
            print(f"  Retrying after a longer delay ({DELAY_BETWEEN_CHUNKS * 2}s) for this chunk...")
            time.sleep(DELAY_BETWEEN_CHUNKS * 2) # Double the delay for timeouts and retry the same chunk.
            continue # Continue to re-process the current chunk after timeout

        except Exception as e: # Catch other unexpected errors
            print(f"  An unexpected error occurred for chunk {actual_idx}: {e}")
            print("  Skipping this chunk and continuing. Consider inspecting the error.")
            continue

print(f"\n Generation complete!")
print(f"   Total Q&A pairs generated : {len(all_qa_pairs)}")

 No checkpoint found — starting fresh

 Starting generation loop...
   Total chunks available : 481
   Chunks to process now  : 15
   Starting from index    : 0
   Model                  : llama-3.1-8b-instant



Generating Q&A pairs:   0%|          | 0/15 [00:00<?, ?it/s]

Generating Q&A pairs:   7%|▋         | 1/15 [00:21<04:56, 21.18s/it]

Generating Q&A pairs:  13%|█▎        | 2/15 [00:43<04:41, 21.69s/it]

Generating Q&A pairs:  20%|██        | 3/15 [01:04<04:19, 21.63s/it]

...

Generating Q&A pairs:  87%|████████▋ | 13/15 [04:38<00:42, 21.29s/it]

Generating Q&A pairs:  93%|█████████▎| 14/15 [04:59<00:21, 21.21s/it]

Generating Q&A pairs: 100%|██████████| 15/15 [05:20<00:00, 21.21s/it]

Generating Q&A pairs: 100%|██████████| 15/15 [05:20<00:00, 21.40s/it]


 Generation complete!
   Total Q&A pairs generated : 149


In [19]:
def _detect_category(question: str) -> str:
    """
    Auto-detect the category of a question based on its label prefix.
    The LLM was instructed to label each question with [HARD FACT], [STRATEGIC], or [CREATIVE].
    """
    q_upper = question.upper()
    if "HARD FACT" in q_upper or "FACT" in q_upper:
        return "hard_fact"
    elif "STRATEGIC" in q_upper or "STRATEG" in q_upper:
        return "strategic"
    elif "CREATIVE" in q_upper or "CREATIVE" in q_upper:
        return "creative"
    else:
        return "general"  # fallback if the LLM didn't label it

# ── Quick category count preview ──
from collections import Counter
cats = Counter(p["category"] for p in all_qa_pairs)
print("Category distribution:")
for cat, count in cats.most_common():
    print(f"   {cat:15s} : {count} pairs")


Category distribution:
   HARD FACT       : 84 pairs
   STRATEGIC       : 38 pairs
   CREATIVE        : 24 pairs
   UNKNOWN         : 3 pairs


# Save as JSONL & Train/Test Split

In [20]:
import random # Added import for random module
from pathlib import Path # Import Path for directory handling

def format_for_finetuning(pair: dict) -> dict:
    """
    Convert a raw Q&A pair into the instruction-following chat format
    that SFTTrainer (Part 2) expects.

    Format:
        system    → sets the persona/role
        user      → the question (with context)
        assistant → the answer
    """
    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are FinanceGPT, an expert financial analyst specializing in "
                    "technology companies. You answer questions based strictly on "
                    "Uber Technologies' 2024 Annual Report. Be precise, factual, and "
                    "professional. Never hallucinate financial figures."
                )
            },
            {
                "role": "user",
                "content": (
                    f"Based on the following passage from Uber's 2024 Annual Report, "
                    f"answer this question:\n\n"
                    f"PASSAGE:\n{pair['context']}\n\n"
                    f"QUESTION: {pair['question']}"
                )
            },
            {
                "role": "assistant",
                "content": pair["answer"]
            }
        ],
        # Keep metadata for reference (not used during training)
        "_meta": {
            "chunk_id" : pair["chunk_id"],
            "category" : pair["category"],
        }
    }


def save_jsonl(data: list[dict], filepath: Path):
    """Save a list of dicts to a .jsonl file (one JSON object per line)."""
    with open(filepath, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"   Saved {len(data):,} records → {filepath}")


# ──────────────────────────────────────────────────────────────────
# SHUFFLE → SPLIT → FORMAT → SAVE
# ──────────────────────────────────────────────────────────────────

# 1. Shuffle so train/test are random, not by chapter order
random.seed(42)  # seed for reproducibility — same shuffle every run
shuffled_pairs = all_qa_pairs.copy()
random.shuffle(shuffled_pairs)

# 2. Calculate split point (80/20)
total      = len(shuffled_pairs)
split_idx  = int(total * 0.80)   # 80% for training

train_raw  = shuffled_pairs[:split_idx]
test_raw   = shuffled_pairs[split_idx:]

print(f"Dataset split:")
print(f"   Total pairs : {total:,}")
print(f"   Train (80%) : {len(train_raw):,}")
print(f"   Test  (20%) : {len(test_raw):,}")

# 3. Format into instruction-following chat format
train_formatted = [format_for_finetuning(p) for p in train_raw]
test_formatted  = [format_for_finetuning(p) for p in test_raw]

# 4. Save to .jsonl files
print("\n Saving files...")
save_jsonl(train_formatted, OUTPUT_DIR / "train.jsonl")
save_jsonl(test_formatted,  OUTPUT_DIR / "golden_test_set.jsonl")

print("\n All files saved!")

Dataset split:
   Total pairs : 149
   Train (80%) : 119
   Test  (20%) : 30

 Saving files...


   Saved 119 records → output/train.jsonl
   Saved 30 records → output/golden_test_set.jsonl

 All files saved!





# Verify & Preview the Output



In [21]:
# ── Read back and verify ──
def load_jsonl(filepath: Path) -> list[dict]:
    """Load a .jsonl file back into a list of dicts."""
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


print("🔍 Verifying saved files...\n")

train_check = load_jsonl(OUTPUT_DIR / "train.jsonl")
test_check  = load_jsonl(OUTPUT_DIR / "golden_test_set.jsonl")

print(f" train.jsonl          → {len(train_check):,} records")
print(f"golden_test_set.jsonl → {len(test_check):,} records")

# ── Preview one record ──
print("\n── Sample record from train.jsonl ──")
sample = train_check[0]
for msg in sample["messages"]:
    role    = msg["role"].upper()
    content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    print(f"\n[{role}]\n{content}")

print(f"\n[METADATA]")
print(f"  Chunk ID : {sample['_meta']['chunk_id']}")
print(f"  Category : {sample['_meta']['category']}")

# ── Category breakdown for both splits ──
print("\n── Category distribution ──")
from collections import Counter
train_cats = Counter(r["_meta"]["category"] for r in train_check)
test_cats  = Counter(r["_meta"]["category"] for r in test_check)

print(f"{'Category':<15} {'Train':>8} {'Test':>8}")
print("-" * 35)
all_cats = set(list(train_cats.keys()) + list(test_cats.keys()))
for cat in sorted(all_cats):
    print(f"{cat:<15} {train_cats.get(cat, 0):>8} {test_cats.get(cat, 0):>8}")


🔍 Verifying saved files...

 train.jsonl          → 119 records
golden_test_set.jsonl → 30 records

── Sample record from train.jsonl ──

[SYSTEM]
You are FinanceGPT, an expert financial analyst specializing in technology companies. You answer questions based strictly on Uber Technologies' 2024 Annual Report. Be precise, factual, and professiona...

[USER]
Based on the following passage from Uber's 2024 Annual Report, answer this question:

PASSAGE:
with the Uber platform overall, which in turn results in broader reach for our Merchants who can attract ...

[ASSISTANT]
[HARD FACT]

[METADATA]
  Chunk ID : 13
  Category : HARD FACT

── Category distribution ──
Category           Train     Test
-----------------------------------
CREATIVE              19        5
HARD FACT             65       19
STRATEGIC             33        5
UNKNOWN                2        1
